# Módulo 8 · Clase 8 — RNN y Transformers: Secuencias y NLP Moderno

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | El problema de las secuencias |
| 2 | RNN simple: memoria a corto plazo |
| 3 | LSTM y GRU: memoria a largo plazo |
| 4 | Embeddings: representar texto |
| 5 | Transformers y atención: el estado del arte |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
np.random.seed(42)
print(f"TensorFlow: {tf.__version__}")

---
## 1. El problema de las secuencias

```
Los datos que hemos visto hasta ahora son 'estáticos':
  imagen → clase, features → precio, ...

Muchos problemas tienen estructura TEMPORAL o SECUENCIAL:
  Texto:             "El gato come" → el orden importa
  Series temporales: temperatura, ventas, bolsa
  Audio:             señal de sonido, speech
  ADN:               secuencia de nucleótidos
  Código:            tokens de programa

El orden importa:
  "perro muerde hombre" ≠ "hombre muerde perro"

Redes densas: tratan todos los inputs como independientes
  → pierden toda la información temporal/posicional

RNN: procesan la secuencia paso a paso,
     manteniendo un 'estado oculto' que actúa como memoria
```

In [ ]:
# Tipos de tareas de secuencias
print("""
Tipos de tareas con secuencias:

  One-to-one:    entrada fija  → salida fija       (clasificación de imágenes)
  One-to-many:   entrada fija  → secuencia         (generación de texto desde imagen)
  Many-to-one:   secuencia     → salida fija        (análisis de sentimiento)
  Many-to-many:  secuencia     → secuencia (misma) (etiquetado POS)
  Many-to-many:  secuencia     → secuencia (dif.)  (traducción automática)

Arquitecturas:
  RNN (Recurrent Neural Network)  → memoria básica
  LSTM (Long Short-Term Memory)   → memoria a largo plazo
  GRU (Gated Recurrent Unit)      → LSTM simplificado, más rápido
  Transformer                     → atención sin recurrencia (estado del arte)
""")

---
## 2. RNN simple: memoria a corto plazo

In [ ]:
# Implementar una RNN simple manualmente
class RNNSimple:
    """
    Para cada paso t:
      h_t = tanh(W_h · h_{t-1} + W_x · x_t + b)

    h_t: estado oculto (la 'memoria')
    x_t: entrada en el paso t
    """
    def __init__(self, n_entrada, n_oculta):
        # Inicialización Glorot
        lim = np.sqrt(6 / (n_entrada + n_oculta))
        self.Wx = np.random.uniform(-lim, lim, (n_entrada, n_oculta))
        self.Wh = np.random.uniform(-lim, lim, (n_oculta,  n_oculta))
        self.b  = np.zeros(n_oculta)
        self.n_oculta = n_oculta

    def forward(self, secuencia):
        """secuencia: (T, n_entrada) → devuelve todos los estados ocultos"""
        T = len(secuencia)
        h = np.zeros(self.n_oculta)  # estado inicial
        estados = []
        for t in range(T):
            h = np.tanh(secuencia[t] @ self.Wx + h @ self.Wh + self.b)
            estados.append(h.copy())
        return np.array(estados), h   # todos los estados, último estado

# Demo: procesar una secuencia de 5 pasos
rnn = RNNSimple(n_entrada=3, n_oculta=4)
secuencia = np.random.randn(5, 3)  # 5 pasos, 3 features cada uno

estados, h_final = rnn.forward(secuencia)
print("RNN simple:")
print(f"  Secuencia: {secuencia.shape}  (T, n_entrada)")
print(f"  Estados:   {estados.shape}    (T, n_oculta)")
print(f"  h_final:   {h_final.round(4)}  ← resume toda la secuencia")

# Visualizar los estados ocultos
fig, ax = plt.subplots(figsize=(9, 4))
for i in range(4):
    ax.plot(estados[:, i], 'o-', lw=2, label=f'h[{i}]')
ax.set_xlabel('Paso de tiempo'); ax.set_ylabel('Activación')
ax.set_title('Evolución del estado oculto h_t en una RNN simple')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# El problema de la RNN simple: gradiente evanescente en secuencias largas
print("Problema: gradiente evanescente en secuencias largas")
print("="*55)

longitudes = [1, 5, 10, 20, 50, 100]
# La derivada de tanh(z) tiene máximo 1 en z=0, pero típicamente ~0.5
# Con una secuencia de T pasos, el gradiente se multiplica T veces por ~0.5
factor = 0.5
for T in longitudes:
    grad = factor ** T
    print(f"  T={T:3d} pasos: gradiente ≈ 0.5^{T} = {grad:.2e}  {'← prácticamente 0' if grad < 1e-6 else ''}")

print("\n→ La RNN 'olvida' lo que ocurrió hace >10-20 pasos")
print("→ Solución: LSTM y GRU con puertas (gates) que controlan el flujo de información")

---
## 3. LSTM y GRU: memoria a largo plazo

```
LSTM (Hochreiter & Schmidhuber, 1997):
  Añade una 'célula de estado' (c_t) que puede:
    - Recordar: puerta de entrada (input gate)
    - Olvidar:  puerta de olvido  (forget gate)
    - Leer:     puerta de salida  (output gate)

  f_t = sigmoid(Wf·[h_{t-1}, x_t] + bf)  ← forget gate: ¿cuánto olvidar?
  i_t = sigmoid(Wi·[h_{t-1}, x_t] + bi)  ← input gate:  ¿qué nuevo guardar?
  g_t = tanh(Wg·[h_{t-1}, x_t] + bg)     ← candidato nuevo contenido
  c_t = f_t * c_{t-1} + i_t * g_t        ← actualizar célula de estado
  o_t = sigmoid(Wo·[h_{t-1}, x_t] + bo)  ← output gate
  h_t = o_t * tanh(c_t)                   ← estado oculto

GRU (Cho et al., 2014):
  Versión simplificada con solo 2 puertas:
    - Puerta de actualización (update gate)
    - Puerta de reseteo (reset gate)
  Menos parámetros → más rápido, casi igual de efectivo

En Keras:
  layers.LSTM(units, return_sequences=False)
  layers.GRU(units, return_sequences=False)
  
  return_sequences=True  → devuelve (T, units) — para apilar RNNs
  return_sequences=False → devuelve (units,)   — solo el último estado
```

In [ ]:
# Demo: capas recurrentes en Keras
# Secuencia de 10 pasos, cada uno con 8 features
x_seq = keras.Input(shape=(10, 8))  # (batch, T, features)

# SimpleRNN
h_rnn  = layers.SimpleRNN(32)(x_seq)
# LSTM
h_lstm = layers.LSTM(32)(x_seq)
# GRU
h_gru  = layers.GRU(32)(x_seq)

print("Salidas con return_sequences=False (solo último estado):")
for nombre, h in [('SimpleRNN', h_rnn), ('LSTM', h_lstm), ('GRU', h_gru)]:
    print(f"  {nombre}: {h.shape}")

# Con return_sequences=True
h_seq = layers.LSTM(32, return_sequences=True)(x_seq)
print(f"\nLSTM con return_sequences=True: {h_seq.shape}  (batch, T, units)")

# Apilar LSTM
h1 = layers.LSTM(64, return_sequences=True)(x_seq)   # devuelve secuencia
h2 = layers.LSTM(32)(h1)                              # consume secuencia, devuelve vector
print(f"\nLSTM apilados: {x_seq.shape} → {h1.shape} → {h2.shape}")

# Parámetros de cada tipo
for nombre, cap in [('SimpleRNN(32)', layers.SimpleRNN(32)),
                     ('GRU(32)',       layers.GRU(32)),
                     ('LSTM(32)',      layers.LSTM(32))]:
    m = keras.Sequential([keras.Input(shape=(10,8)), cap])
    print(f"  {nombre}: {m.count_params():,} parámetros")

In [ ]:
# Aplicación: predicción de series temporales (temperatura sintética)
# Generar serie: temperatura con tendencia estacional
T = 500
t = np.linspace(0, 10 * np.pi, T)
temperatura = np.sin(t) + 0.5*np.sin(3*t) + 0.3*np.random.randn(T)

# Crear ventanas de entrada/salida
N_PASOS = 20  # predecir con 20 pasos anteriores
X_ts = np.array([temperatura[i:i+N_PASOS] for i in range(len(temperatura)-N_PASOS-1)])
y_ts = temperatura[N_PASOS+1:]

X_ts = X_ts[..., np.newaxis]  # añadir dimensión feature: (N, 20, 1)

split = int(0.8 * len(X_ts))
X_tr_ts, X_te_ts = X_ts[:split], X_ts[split:]
y_tr_ts, y_te_ts = y_ts[:split], y_ts[split:]

print(f"Serie temporal:")
print(f"  Muestras train: {X_tr_ts.shape}")
print(f"  Muestras test:  {X_te_ts.shape}")

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(temperatura, lw=1, color='steelblue')
ax.axvline(split + N_PASOS, color='red', ls='--', label='train/test split')
ax.set_title('Serie temporal de temperatura sintética')
ax.set_xlabel('Tiempo'); ax.set_ylabel('Temperatura'); ax.legend()
ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Comparar: Dense vs LSTM para predicción de series temporales
tf.random.set_seed(42)

# LSTM
m_lstm = keras.Sequential([
    layers.Input(shape=(N_PASOS, 1)),
    layers.LSTM(64, return_sequences=True),
    layers.LSTM(32),
    layers.Dense(16, activation='relu'),
    layers.Dense(1),
], name='LSTM_ts')
m_lstm.compile('adam', 'mse', ['mae'])
h_lstm = m_lstm.fit(X_tr_ts, y_tr_ts, epochs=30, batch_size=32,
                    validation_split=0.1, verbose=0)
_, mae_lstm = m_lstm.evaluate(X_te_ts, y_te_ts, verbose=0)

# Dense (sin memoria temporal)
m_dense_ts = keras.Sequential([
    layers.Input(shape=(N_PASOS, 1)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1),
], name='Dense_ts')
m_dense_ts.compile('adam', 'mse', ['mae'])
m_dense_ts.fit(X_tr_ts, y_tr_ts, epochs=30, batch_size=32,
               validation_split=0.1, verbose=0)
_, mae_dense = m_dense_ts.evaluate(X_te_ts, y_te_ts, verbose=0)

print(f"Dense → MAE: {mae_dense:.4f}")
print(f"LSTM  → MAE: {mae_lstm:.4f}")

# Visualizar predicciones
y_pred_lstm  = m_lstm.predict(X_te_ts, verbose=0).ravel()
y_pred_dense = m_dense_ts.predict(X_te_ts, verbose=0).ravel()

fig, ax = plt.subplots(figsize=(12, 4))
x_plot = range(len(y_te_ts))
ax.plot(x_plot, y_te_ts,     'k-',        lw=1.5, label='Real', alpha=0.7)
ax.plot(x_plot, y_pred_lstm, 'steelblue', lw=1.5, label=f'LSTM (MAE={mae_lstm:.3f})')
ax.plot(x_plot, y_pred_dense,'coral',     lw=1.5, label=f'Dense (MAE={mae_dense:.3f})', alpha=0.7)
ax.set_title('Predicción de serie temporal: Dense vs LSTM')
ax.set_xlabel('Tiempo'); ax.set_ylabel('Temperatura')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 4. Embeddings: representar texto

In [ ]:
print("""
¿Cómo convertir texto a números?

  One-hot: vocabulario de 10000 palabras → vector de 10000 ceros con un 1
    Problema: 10000 dimensiones, sin similitud semántica
    "rey" y "reina" son tan distantes como "rey" y "silla"

  Embedding: mapear cada palabra a un vector denso de 128 dimensiones
    Los vectores se aprenden durante el entrenamiento
    Palabras similares quedan cerca en el espacio:
      vec(rey) - vec(hombre) + vec(mujer) ≈ vec(reina)  [Word2Vec]

layers.Embedding(vocab_size, embed_dim):
  vocab_size: número de tokens distintos
  embed_dim:  dimensión del vector (típico: 64-512)
  Input: (batch, T) → enteros (ids de tokens)
  Output: (batch, T, embed_dim)
""")

# Demostración: Embedding layer
VOCAB = 100
EMBED = 16
SEQ_LEN = 10

embedding = layers.Embedding(VOCAB, EMBED)
# Simular una secuencia de tokens (enteros 0-99)
tokens = tf.constant([[5, 12, 3, 45, 0, 7, 89, 22, 4, 1]])
vectors = embedding(tokens)
print(f"Input tokens: {tokens.shape}  (batch, seq_len)")
print(f"Output embeddings: {vectors.shape}  (batch, seq_len, embed_dim)")
print(f"\nVector del token 5: {vectors[0,0,:].numpy().round(3)}")

In [ ]:
# Análisis de sentimiento con IMDB: reseñas de cine → positivo/negativo
MAX_FEATURES = 10000  # vocabulario: las 10000 palabras más frecuentes
MAX_LEN      = 200    # recortar/rellenar todas las reseñas a 200 tokens

(X_tr_imdb, y_tr_imdb), (X_te_imdb, y_te_imdb) = keras.datasets.imdb.load_data(
    num_words=MAX_FEATURES
)

print(f"Train: {len(X_tr_imdb)} reseñas")
print(f"Ejemplo — tokens: {X_tr_imdb[0][:15]}...")
print(f"Etiqueta: {y_tr_imdb[0]} (1=positivo, 0=negativo)")

# Pad/truncar a MAX_LEN
X_tr_imdb = keras.utils.pad_sequences(X_tr_imdb, maxlen=MAX_LEN)
X_te_imdb = keras.utils.pad_sequences(X_te_imdb, maxlen=MAX_LEN)
print(f"\nDespués de pad: {X_tr_imdb.shape}")

In [ ]:
# Modelo: Embedding + LSTM + clasificador
tf.random.set_seed(42)
modelo_nlp = keras.Sequential([
    layers.Input(shape=(MAX_LEN,)),
    layers.Embedding(MAX_FEATURES, 64),          # (N, 200) → (N, 200, 64)
    layers.LSTM(64, return_sequences=True),       # (N, 200, 64) → (N, 200, 64)
    layers.LSTM(32),                              # (N, 200, 64) → (N, 32)
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid'),        # binario: pos/neg
], name='LSTM_sentiment')

modelo_nlp.summary()

modelo_nlp.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

h_nlp = modelo_nlp.fit(
    X_tr_imdb, y_tr_imdb,
    epochs=5, batch_size=128,
    validation_split=0.1,
    verbose=1
)

_, acc_nlp = modelo_nlp.evaluate(X_te_imdb, y_te_imdb, verbose=0)
print(f"\nTest accuracy (IMDB): {acc_nlp:.4f} ({acc_nlp:.2%})")

---
## 5. Transformers: el estado del arte

```
Problema de las RNN:
  1. Procesamiento secuencial → lento en GPU
  2. Incluso con LSTM, secuencias muy largas son difíciles
  3. Para traducir "the animal didn't cross the street because it was tired"
     ¿"it" se refiere a "animal" o "street"? Muy lejos en la secuencia.

Transformer (Vaswani et al., "Attention Is All You Need", 2017):
  - Procesa TODOS los tokens EN PARALELO (no secuencialmente)
  - Mecanismo de ATENCIÓN: cada token puede 'atender' a cualquier otro
  - Permite relaciones a cualquier distancia con igual coste

Atención (Attention):
  Attention(Q, K, V) = softmax(Q·Kᵀ / √d_k) · V

  Q (Query):  ¿qué busca este token?
  K (Key):    ¿qué ofrece cada token?
  V (Value):  ¿qué información contiene cada token?

  El softmax convierte las similitudes en pesos de atención.
  Cada token pondera la información de TODOS los tokens según relevancia.

Modelos basados en Transformers:
  BERT (Google, 2018)    → comprensión de texto
  GPT-2/3/4 (OpenAI)    → generación de texto
  T5, LLaMA, Mistral    → multiusos
  ViT (Vision Transformer) → imágenes
```

In [ ]:
# Visualizar cómo funciona la atención
np.random.seed(42)

frase = ['El', 'gato', 'no', 'cruzó', 'la', 'calle', 'porque', 'estaba', 'cansado']
T = len(frase)

# Simular pesos de atención para el token 'estaba' (idx 7)
# En realidad se aprenden; aquí los creamos a mano para ilustrar
pesos_atencion = np.array([
    [0.05, 0.35, 0.02, 0.03, 0.02, 0.03, 0.05, 0.15, 0.30],  # 'estaba' → atiende 'gato' y 'cansado'
    [0.20, 0.05, 0.03, 0.15, 0.02, 0.03, 0.05, 0.25, 0.22],  # 'cruzó' → atiende 'gato' y 'estaba'
    [0.05, 0.05, 0.05, 0.05, 0.05, 0.40, 0.05, 0.15, 0.15],  # 'cansado' → atiende 'calle'
])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
tokens_consultados = ['estaba', 'cruzó', 'cansado']
idx_consultados = [7, 3, 8]

for ax, (token_q, pesos) in zip(axes, zip(tokens_consultados, pesos_atencion)):
    bars = ax.barh(range(T), pesos, color=['red' if i == idx_consultados[tokens_consultados.index(token_q)]
                   else 'steelblue' for i in range(T)], alpha=0.8)
    ax.set_yticks(range(T)); ax.set_yticklabels(frase, fontsize=11)
    ax.set_xlabel('Peso de atención')
    ax.set_title(f'Token "{token_q}" presta atención a:')
    ax.grid(True, alpha=0.3, axis='x')
    for bar, peso in zip(bars, pesos):
        ax.text(peso + 0.005, bar.get_y() + bar.get_height()/2,
                f'{peso:.2f}', va='center', fontsize=8)

plt.suptitle('Mecanismo de atención (simulado)', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Multi-Head Attention en Keras
print("Multi-Head Attention en Keras:")
x = keras.Input(shape=(MAX_LEN, 64))  # (batch, seq, embed)
att = layers.MultiHeadAttention(num_heads=4, key_dim=16)(x, x)
print(f"  Input:  {x.shape}")
print(f"  Output: {att.shape}")

# Transformer block básico para clasificación de sentimiento
print("\nConstructor de un bloque Transformer:")
print("""
def transformer_block(x, embed_dim, num_heads, ff_dim, dropout=0.1):
    # Multi-Head Self-Attention
    attn = layers.MultiHeadAttention(num_heads, key_dim=embed_dim//num_heads)(x, x)
    attn = layers.Dropout(dropout)(attn)
    x    = layers.LayerNormalization()(x + attn)   # residual + norm

    # Feed-Forward Network
    ff  = layers.Dense(ff_dim, activation='relu')(x)
    ff  = layers.Dense(embed_dim)(ff)
    ff  = layers.Dropout(dropout)(ff)
    out = layers.LayerNormalization()(x + ff)       # residual + norm
    return out
""")

def transformer_block(x, embed_dim, num_heads, ff_dim, dropout=0.1):
    attn = layers.MultiHeadAttention(num_heads, key_dim=embed_dim//num_heads)(x, x)
    attn = layers.Dropout(dropout)(attn)
    x    = layers.LayerNormalization()(x + attn)
    ff   = layers.Dense(ff_dim, activation='relu')(x)
    ff   = layers.Dense(embed_dim)(ff)
    ff   = layers.Dropout(dropout)(ff)
    return layers.LayerNormalization()(x + ff)

# Modelo Transformer para IMDB
tf.random.set_seed(42)
EMBED_DIM = 64

entradas = keras.Input(shape=(MAX_LEN,))
x = layers.Embedding(MAX_FEATURES, EMBED_DIM)(entradas)
x = transformer_block(x, EMBED_DIM, num_heads=4, ff_dim=128)
x = layers.GlobalAveragePooling1D()(x)  # resumir toda la secuencia
x = layers.Dropout(0.3)(x)
x = layers.Dense(32, activation='relu')(x)
x = layers.Dropout(0.2)(x)
salidas = layers.Dense(1, activation='sigmoid')(x)

modelo_transformer = keras.Model(entradas, salidas, name='Transformer_IMDB')
modelo_transformer.compile('adam', 'binary_crossentropy', ['accuracy'])
modelo_transformer.summary()

h_tr = modelo_transformer.fit(
    X_tr_imdb, y_tr_imdb,
    epochs=5, batch_size=128,
    validation_split=0.1,
    verbose=1
)

_, acc_tr = modelo_transformer.evaluate(X_te_imdb, y_te_imdb, verbose=0)
print(f"\nTransformer — Test accuracy: {acc_tr:.4f}")
print(f"LSTM        — Test accuracy: {acc_nlp:.4f}")

---
## Práctica

### Ejercicio 1 — GRU para IMDB

**Enunciado:** Las GRU son más rápidas que las LSTM.

1. Construye el mismo modelo de análisis de sentimiento pero con GRU en lugar de LSTM
2. Compara tiempo de entrenamiento y accuracy con el modelo LSTM
3. ¿Merece la pena la complejidad extra del LSTM?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import time
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

MAX_FEATURES, MAX_LEN = 10000, 200
(X_tr, y_tr), (X_te, y_te) = keras.datasets.imdb.load_data(num_words=MAX_FEATURES)
X_tr = keras.utils.pad_sequences(X_tr, maxlen=MAX_LEN)
X_te = keras.utils.pad_sequences(X_te, maxlen=MAX_LEN)

def build_rnn(tipo='lstm'):
    rnn_capa = layers.LSTM(64, return_sequences=True) if tipo == 'lstm' else layers.GRU(64, return_sequences=True)
    rnn_capa2 = layers.LSTM(32) if tipo == 'lstm' else layers.GRU(32)
    return keras.Sequential([
        layers.Input(shape=(MAX_LEN,)),
        layers.Embedding(MAX_FEATURES, 64),
        rnn_capa, rnn_capa2,
        layers.Dense(16, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid'),
    ])

resultados = {}
for tipo in ['lstm', 'gru']:
    tf.random.set_seed(42)
    m = build_rnn(tipo)
    m.compile('adam', 'binary_crossentropy', ['accuracy'])
    t0 = time.time()
    m.fit(X_tr, y_tr, epochs=3, batch_size=128, validation_split=0.1, verbose=0)
    t1 = time.time()
    _, acc = m.evaluate(X_te, y_te, verbose=0)
    resultados[tipo] = {'acc': acc, 'tiempo': t1-t0, 'params': m.count_params()}
    print(f"{tipo.upper()}: acc={acc:.4f}, tiempo={t1-t0:.1f}s, params={m.count_params():,}")

print(f"\nGRU es {resultados['lstm']['tiempo']/resultados['gru']['tiempo']:.1f}x más rápido")
print(f"Diferencia en accuracy: {abs(resultados['lstm']['acc']-resultados['gru']['acc']):.4f}")
```
</details>

---
### Ejercicio 2 — Predicción multivariante

**Enunciado:** La serie temporal anterior solo tenía 1 feature. Predice usando múltiples variables:

1. Genera: temperatura + humedad (correlacionadas + ruido)
2. Usa ventanas de 20 pasos con 2 features → predice la temperatura
3. Compara: LSTM con 1 feature vs LSTM con 2 features
4. ¿Ayuda conocer la humedad?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

T = 600
t = np.linspace(0, 12*np.pi, T)
temp     = np.sin(t) + 0.5*np.sin(3*t) + 0.3*np.random.randn(T)
humedad  = -0.6*np.sin(t) + 0.3*np.random.randn(T) + 0.5  # correlacionada inversamente

# Normalizar
temp_n    = (temp - temp.mean()) / temp.std()
humedad_n = (humedad - humedad.mean()) / humedad.std()

N_PASOS = 20
def crear_ventanas(features, target, n):
    X = np.array([features[i:i+n] for i in range(len(target)-n-1)])
    y = target[n+1:]
    return X, y

# 1 feature
X1 = temp_n[:, None]
X_ts1, y_ts = crear_ventanas(X1, temp_n, N_PASOS)

# 2 features
X2 = np.stack([temp_n, humedad_n], axis=-1)
X_ts2, _   = crear_ventanas(X2, temp_n, N_PASOS)

split = int(0.8 * len(y_ts))

maes = {}
for nombre, X_ts in [('1 feature (temp)', X_ts1), ('2 features (temp+humedad)', X_ts2)]:
    tf.random.set_seed(42)
    n_feat = X_ts.shape[-1]
    m = keras.Sequential([
        layers.Input(shape=(N_PASOS, n_feat)),
        layers.LSTM(64, return_sequences=True),
        layers.LSTM(32),
        layers.Dense(1),
    ])
    m.compile('adam', 'mse', ['mae'])
    m.fit(X_ts[:split], y_ts[:split], epochs=25, batch_size=32,
          validation_split=0.1, verbose=0)
    _, mae = m.evaluate(X_ts[split:], y_ts[split:], verbose=0)
    maes[nombre] = mae
    print(f"{nombre}: MAE={mae:.4f}")

print(f"\nMejora: {maes['1 feature (temp)'] - maes['2 features (temp+humedad)']:.4f}")
```
</details>

---
### Ejercicio 3 — Embedding personalizado

**Enunciado:** Visualiza qué aprenden los embeddings.

1. Entrena el modelo LSTM de IMDB (ya entrenado)
2. Extrae los pesos del Embedding layer: `modelo.layers[1].get_weights()[0]`
3. Usa PCA para reducir a 2D
4. Visualiza las 20 palabras más frecuentes en el espacio 2D
5. ¿Palabras similares quedan cerca?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Diccionario inverso IMDB
word_index = keras.datasets.imdb.get_word_index()
idx2word   = {v+3: k for k, v in word_index.items()}
idx2word.update({0:'<PAD>', 1:'<START>', 2:'<UNK>', 3:'<UNUSED>'})

# Extraer embeddings
embed_pesos = modelo_nlp.layers[1].get_weights()[0]  # (10000, 64)
print(f"Pesos del Embedding: {embed_pesos.shape}")

# Las palabras más frecuentes son las de índice más bajo (IMDB las ordena por frecuencia)
n_show = 40
indices = list(range(4, 4 + n_show))
vectores = embed_pesos[indices]  # (40, 64)
palabras = [idx2word.get(i, '?') for i in indices]

# PCA a 2D
pca = PCA(n_components=2)
coords = pca.fit_transform(vectores)  # (40, 2)

fig, ax = plt.subplots(figsize=(12, 9))
ax.scatter(coords[:,0], coords[:,1], c='steelblue', s=60, zorder=3)
for (x, y), palabra in zip(coords, palabras):
    ax.annotate(palabra, (x, y), textcoords='offset points',
                xytext=(5, 5), fontsize=9)
ax.set_title('Embeddings de IMDB — PCA 2D (40 palabras más frecuentes)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Varianza explicada por PC1+PC2: {pca.explained_variance_ratio_.sum():.2%}")
```
</details>

---
### Ejercicio 4 — LSTM bidireccional

**Enunciado:** Un LSTM lee la secuencia de izquierda a derecha. Un LSTM bidireccional la lee en ambas direcciones y concatena los resultados.

```python
layers.Bidirectional(layers.LSTM(64))
```

1. Adapta el modelo de análisis de sentimiento para usar LSTM bidireccional
2. ¿Mejora la accuracy en IMDB?
3. ¿Cuántos parámetros extra tiene?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

MAX_FEATURES, MAX_LEN = 10000, 200
(X_tr, y_tr), (X_te, y_te) = keras.datasets.imdb.load_data(num_words=MAX_FEATURES)
X_tr = keras.utils.pad_sequences(X_tr, maxlen=MAX_LEN)
X_te = keras.utils.pad_sequences(X_te, maxlen=MAX_LEN)

modelos = {
    'LSTM unidireccional': keras.Sequential([
        layers.Input(shape=(MAX_LEN,)),
        layers.Embedding(MAX_FEATURES, 64),
        layers.LSTM(64, return_sequences=True),
        layers.LSTM(32),
        layers.Dense(16, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid'),
    ]),
    'LSTM bidireccional': keras.Sequential([
        layers.Input(shape=(MAX_LEN,)),
        layers.Embedding(MAX_FEATURES, 64),
        layers.Bidirectional(layers.LSTM(64, return_sequences=True)),
        layers.Bidirectional(layers.LSTM(32)),
        layers.Dense(16, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid'),
    ]),
}

print(f"{'Modelo':25} {'Params':>12} {'Test acc':>10}")
print('-' * 52)
for nombre, m in modelos.items():
    tf.random.set_seed(42)
    m.compile('adam', 'binary_crossentropy', ['accuracy'])
    m.fit(X_tr, y_tr, epochs=3, batch_size=128, validation_split=0.1, verbose=0)
    _, acc = m.evaluate(X_te, y_te, verbose=0)
    print(f"{nombre:25} {m.count_params():>12,} {acc:>10.4f}")
```
</details>

---
### Ejercicio 5 (reto) — Generación de texto

**Enunciado:** Entrena un modelo para generar texto carácter a carácter.

1. Usa un texto corto (poema, canción, código — mínimo 1000 caracteres)
2. Codifica carácter → entero
3. Modelo: Embedding(vocab) → LSTM(128) → Dense(vocab, softmax)
4. Entrena para predecir el siguiente carácter
5. Genera texto nuevo con muestreo (no argmax — usa temperatura=0.8)

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

# Texto de ejemplo (fragmento de "En el principio" — Génesis simplificado)
texto = """
en el principio creo dios los cielos y la tierra la tierra estaba desordenada
y vacia y las tinieblas estaban sobre la faz del abismo y el espiritu de dios
se movia sobre la faz de las aguas y dijo dios sea la luz y fue la luz y vio
dios que la luz era buena y separo dios la luz de las tinieblas y llamo dios a
la luz dia y a las tinieblas llamo noche y fue la tarde y la manana el primer dia
y dijo dios haya expansion en medio de las aguas y separe las aguas de las aguas
e hizo dios la expansion y separo las aguas que estaban debajo de la expansion
de las aguas que estaban sobre la expansion y fue asi y llamo dios a la expansion
cielos y fue la tarde y la manana el segundo dia y dijo dios junense las aguas
que estan debajo de los cielos en un lugar y descubrase lo seco y fue asi y llamo
dios a lo seco tierra y a la reunion de las aguas llamo mares y vio dios que era bueno
""" * 5

# Vocabulario de caracteres
chars  = sorted(set(texto))
vocab  = len(chars)
c2i    = {c:i for i,c in enumerate(chars)}
i2c    = {i:c for c,i in c2i.items()}
encoded = np.array([c2i[c] for c in texto])

print(f"Texto: {len(texto)} caracteres, {vocab} únicos")

# Crear secuencias
SEQ_LEN = 40
X = np.array([encoded[i:i+SEQ_LEN] for i in range(len(encoded)-SEQ_LEN-1)])
y = encoded[SEQ_LEN+1:]

# Modelo
tf.random.set_seed(42)
m = keras.Sequential([
    layers.Input(shape=(SEQ_LEN,)),
    layers.Embedding(vocab, 32),
    layers.LSTM(128, return_sequences=True),
    layers.LSTM(64),
    layers.Dense(vocab, activation='softmax'),
])
m.compile('adam', 'sparse_categorical_crossentropy')
m.fit(X, y, epochs=30, batch_size=128, verbose=0)
print("Entrenamiento completado")

def generar(modelo, semilla, n_chars=200, temperatura=0.8):
    semilla_encoded = np.array([c2i.get(c, 0) for c in semilla[-SEQ_LEN:]])
    resultado = list(semilla)
    for _ in range(n_chars):
        x = semilla_encoded[-SEQ_LEN:][None, :]  # (1, SEQ_LEN)
        probs = modelo.predict(x, verbose=0)[0]   # (vocab,)
        probs = np.log(probs + 1e-8) / temperatura
        probs = np.exp(probs - probs.max())
        probs /= probs.sum()
        siguiente = np.random.choice(vocab, p=probs)
        resultado.append(i2c[siguiente])
        semilla_encoded = np.append(semilla_encoded, siguiente)
    return ''.join(resultado)

print("\n=== Texto generado ===")
print(generar(m, semilla='en el principio ', n_chars=300, temperatura=0.8))
```
</details>

---
## Resumen

```python
# RNN para secuencias
modelo = keras.Sequential([
    layers.Input(shape=(T, n_features)),  # (batch, pasos, features)
    layers.LSTM(64, return_sequences=True),  # apilado → True
    layers.LSTM(32),                          # último → False
    layers.Dense(n_salidas),
])

# NLP: texto → embedding → RNN
modelo = keras.Sequential([
    layers.Input(shape=(MAX_LEN,)),         # (batch, T) enteros
    layers.Embedding(VOCAB, embed_dim),     # (batch, T, embed_dim)
    layers.LSTM(64),                        # o GRU, o Bidirectional(LSTM())
    layers.Dense(1, activation='sigmoid'),  # o softmax para multiclase
])

# Transformer block
layers.MultiHeadAttention(num_heads=4, key_dim=16)(x, x)
layers.GlobalAveragePooling1D()  # resumir secuencia a vector

# Tipos de capas recurrentes
# SimpleRNN(n)          → básico, olvida rápido
# LSTM(n)               → memoria larga, más parámetros
# GRU(n)                → equilibrio velocidad/rendimiento
# Bidirectional(LSTM(n)) → lee secuencia en ambas direcciones
```

## ¿Qué viene en la próxima clase?
En la **Clase 9** aprenderemos a **monitorizar el entrenamiento**: TensorBoard, learning rate schedules, y técnicas avanzadas de debugging de modelos.